# SI 618 WN Project Part III

## Team Members

#### Carlos Figueredo (carlosfc), James Zhu (jazhu)


## Predicting Rurality Level of U.S. Counties based on Demographics

## Overview

Demographic trends in both developed and developing countries are shifting, with notable changes in family size, age distribution, and racial composition. In the United States, analyzing demographic information such as race and age alongside rurality levels provides valuable insights into current birth trends and their broader implications. This project examines how birth counts vary across different degrees of urbanization in U.S. counties, considering the influence of demographic factors. Understanding these relationships is essential for anticipating challenges like labor shortages and reduced economic innovation, as rural and urban areas may experience distinct natality patterns shaped by their unique demographic profiles.


## Data Sources

For this project, we use **3** data sources. As of 2025, all the data is public and accessible through the web. 

The first data source is the U.S. Centers for Disease Control and Prevention ([CDC](https://www.cdc.gov/)). Specifically, we use data obtained from the [Natality Information Database](https://wonder.cdc.gov/controller/datarequest/D66). From this database, we extract the birth information for the years 2011 to 2013 for all available counties in the United States. This information is stored in the files `<year>.txt` where `<year>` is replaced by the respective year period. This database provides information at the level of counties, which gives us a high granularity of birth information in the U.S.

The second data source is the U.S. Department of Agriculture ([USDA](https://www.usda.gov/)). Particulary, we use the **2013 Rural-Urban Continuum Codes** available in this [link](https://www.ers.usda.gov/data-products/rural-urban-continuum-codes). All the information is provided by the Economic Research Services Division of the USDA. The information can be downloaded as an Excel file (`.xsl` format), and it is available in the `ruralurbancodes2013.xls` file. This dataset also provides information at the level of counties, and it contains counties' degree of urbanization. Metropolitan counties are categorized by their population size, and nonmetropolitan counties are categorized by their degree of urbanization and adjacency to a metro area.

The third data source is the National Cancer Institute. We use the [County Population Data Dictionary](https://seer.cancer.gov/popdata/popdic.html), which contains subpopulation counts aggregated by race, sex, age groups and origin. This dataset provides for each county a powerful tool for analyzing diversity mixes.

These datasets provide us a joint county-level view of natality, demographics and urbanization.

NOTE: To access the birth count data, visit [https://wonder.cdc.gov/natality-current.html](https://wonder.cdc.gov/natality-current.html). You will need to click "I agree" at the bottom of the page to accept the terms of usage.  This will take you to the page where you can download the data. 

In [129]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import gzip

# Preprocessing Data

### Birth Counts Dataset

For our first dataset, we need to read the information for 2011, 2012, and 2013. Next, we focus our attention in the Annual Birth Counts, County Code, and the referenced year.

In [130]:
def read_births():
    births = pd.DataFrame()
    for y in range(2011, 2014):
        da = pd.read_csv("%4d.txt" % y, delimiter="\t", dtype={"County Code": object})
        da = da[["County", "County Code", "Births"]]
        da["year"] = y
        births = pd.concat([births, da])
    births = births.dropna()
    births = births.rename({'County Code':'FIPS'}, axis=1)
    unidentified_mask = births['County'].str.contains('Unidentified')
    births = births[~ unidentified_mask]
    births = births.groupby(['FIPS']).agg(Births=('Births', 'mean'),
                                 StdBirths=('Births', 'std'))
    return births

births = read_births()
births.info()

<class 'pandas.core.frame.DataFrame'>
Index: 524 entries, 01003 to 55139
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Births     524 non-null    float64
 1   StdBirths  524 non-null    float64
dtypes: float64(2)
memory usage: 12.3+ KB


Notice that the births dataset provides restricted information because it only covers 524 different counties.

### Demographics Dataset

For our second dataset, we read the information for the year 2016. Then, we read the fixed width formatted text file to extract the different demographic features. We focus our attention on ```Race``` and ```Age``` because they can discriminate better over the rurality level. Generally, ```Sex``` is similar for **Male** and **Female**, so we don't consider this demographic feature.

In [131]:
def read_demog():
    grid = [1, 5, 7, 9, 12, 14, 15, 16, 17, 19, 27]
    ranges = [(grid[i]-1, grid[i+1]-1) for i in range(len(grid)-1)]
    with gzip.open("2016ages.txt.gz") as io:
        demog = pd.read_fwf(io, colspecs=ranges, header=None)
    demog.columns = ["Year", "State", "StateFIPS", "CountyFIPS", "Registry",
                    "Race", "Origin", "Sex", "Age", "Population"]
    demog["FIPS"] = ["%02d%03d" % (x, y) for (x, y) in zip(demog.StateFIPS, demog.CountyFIPS)]
    demog = demog[["FIPS", "Race", "Age", "Population"]]
    demog["Race"] = demog["Race"].replace([1, 2, 3, 4], ["W", "B", "N", "A"]) # White/Black/Native/Asian
    # Age group labels (For Reference)
    age_groups = ["0", "1-4", "5-9", "10-14", "15-19", "20-24", "25-29", "30-34", "35-39",
                "40-44", "45-49", "50-54", "55-59", "60-64", "65-69", "70-74", "75-79",
                "80-84", "85-89", "90+"]
    return demog

def pivot_demog(demog):
    demog = demog.pivot_table(index="FIPS", columns=["Race", "Age"], values="Population", aggfunc="sum")
    na = demog.columns.tolist()
    demog.columns = ["%s:%d" % tuple(x) for x in na]
    demog = demog.fillna(0)
    return demog

demog = read_demog()
demog = pivot_demog(demog)
print(f"Number of Different Counties: {len(demog)}")
demog.head()


Number of Different Counties: 3142


,A:0,A:1,A:2,A:3,A:4,A:5,A:6,A:7,A:8,A:9,...,W:10,W:11,W:12,W:13,W:14,W:15,W:16,W:17,W:18,W:19
FIPS,,,,,,,,,,,,,,,,,,,,,
01001,12.0,59.0,55.0,64.0,62.0,61.0,45.0,66.0,88.0,74.0,...,3270.0,3320.0,3151.0,2544.0,2278.0,1829.0,1333.0,918.0,443.0,199.0
01003,28.0,135.0,176.0,233.0,313.0,181.0,179.0,149.0,267.0,195.0,...,12486.0,13048.0,13598.0,13186.0,13052.0,9885.0,6836.0,4286.0,2401.0,1247.0
01005,3.0,4.0,13.0,10.0,10.0,12.0,16.0,14.0,16.0,14.0,...,866.0,936.0,940.0,960.0,980.0,823.0,527.0,365.0,194.0,101.0
01007,0.0,5.0,2.0,7.0,9.0,4.0,14.0,8.0,7.0,4.0,...,1282.0,1337.0,1205.0,1093.0,1042.0,771.0,577.0,352.0,204.0,90.0
01009,4.0,26.0,33.0,19.0,22.0,20.0,16.0,15.0,19.0,22.0,...,3938.0,3974.0,3824.0,3629.0,3409.0,2602.0,1772.0,1185.0,634.0,316.0


Notice that this dataset is much richer in terms of counties because it contains 3142 different U.S. counties.

The codes in the columns above has the following meanings.
#### Race
| A     | B     | N      | W     |
|-------|-------|--------|-------|
| Asian | Black | Native | White |

#### Age Group Label

|Age Label|0|1|2|3|4|5|6|7|8|9|
|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|
|Years | 0     | 1-4   | 5-9   | 10-14 | 15-19 | 20-24 | 25-29 | 30-34 | 35-39 | 40-44 |

$\quad$

| 10    | 11    | 12    | 13    | 14    | 15    | 16    | 17    | 18    | 19   |
|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|
| 45-49 | 50-54 | 55-59 | 60-64 | 65-69 | 70-74 | 75-79 | 80-84 | 85-89 | 90+ |

### Rural Codes Dataset

Now, we focus our attention in the third dataset. This contains information about urbanization and rurality levels for counties in the U.S. We read this data from an excel file, for which we need the `xlrd` library. This contains the **target** variable for our classification problem.

In [132]:
def read_rucc():
    rucc = pd.read_excel("ruralurbancodes2013.xls", sheet_name=None) # Requires xlrd. Run pip install xlrd
    rucc = rucc["Rural-urban Continuum Code 2013"] # Get first sheet
    rucc["FIPS"] = ["%05d" % x for x in rucc.FIPS] # FIPS to object dtype
    rucc['RUCC_Category'] = rucc['RUCC_2013'].apply(lambda x: "Metro" if x <= 3 else "Nonmetro")
    rucc = rucc.dropna()
    return rucc

rucc = read_rucc()
print(f"Number of Different Counties: {rucc["FIPS"].nunique()}")
rucc.head()

Number of Different Counties: 3232


,FIPS,State,County_Name,Population_2010,RUCC_2013,Description,RUCC_Category
0,01001,AL,Autauga County,54571,2.0,"Metro - Counties in metro areas of 250,000 to ...",Metro
1,01003,AL,Baldwin County,182265,3.0,Metro - Counties in metro areas of fewer than ...,Metro
2,01005,AL,Barbour County,27457,6.0,"Nonmetro - Urban population of 2,500 to 19,999...",Nonmetro
3,01007,AL,Bibb County,22915,1.0,Metro - Counties in metro areas of 1 million p...,Metro
4,01009,AL,Blount County,57322,1.0,Metro - Counties in metro areas of 1 million p...,Metro


This dataset contains information about 3232 different U.S. Counties.

## Data Manipulation

We focus our attention in merging all the data sources.

Since rurality can be easily predicted by using popularity, we normalize the counts by populations. Specifically, the subpopulation demographics are normalized. Birth counts are also normalied.

In [142]:
def read_data(drop_FIPS=True):
    births = read_births()
    demog = read_demog()
    demog = pivot_demog(demog)
    demog_sum = demog.sum(axis=1).to_frame()
    demog_sum.columns = ["Population"]
    births_demog = pd.merge(births, demog_sum, on="FIPS")
    births_demog["Births"] = births_demog["Births"] / births_demog["Population"]
    births_demog["StdBirths"] = births_demog["StdBirths"] / births_demog["Population"]
    births_demog = births_demog.drop(columns=["Population"])
    norm_demog = demog.div(demog.sum(axis=1), axis=0)
    df = pd.merge(births_demog, norm_demog, on="FIPS", how="right")
    rucc = read_rucc()[["FIPS", "RUCC_Category"]]
    rucc["Metro"] = rucc["RUCC_Category"].apply(lambda x: 1 if x == "Metro" else 0)
    rucc = rucc.drop(columns=["RUCC_Category"])
    df = pd.merge(df, rucc, on="FIPS")
    if drop_FIPS:
        df = df.drop(columns=["FIPS"])
    return df

df = read_data()

In [148]:
print(f"Number of Different Counties: {df.shape[0]}")
df.isna().sum()[df.isna().sum() > 0]

Number of Different Counties: 3141


Births       2617
StdBirths    2617
dtype: int64

Notice that a big portion of the merged data contains missing information (2617 out of 3141 counties in df).

## Imputation of Births

We impute the value ```Births```, and ```StdBirths``` by labelling as unknown the missing values. Additionally, we set as high or low, dependeing on the median values of these features.

In [149]:
def impute(row, column):
    if pd.isna(row[column]):
        return 'Unknown'
    elif row[column] > df[column].median():
        return 'High'
    else:
        return 'Low'

df['Births'] = df.apply(lambda row: impute(row, 'Births'), axis=1)
df['StdBirths'] = df.apply(lambda row: impute(row, 'StdBirths'), axis=1)

In [150]:
df.isna().sum().sum()

0

# Clasification Analysis

In [137]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [138]:
X = df.drop(columns=["Metro"])
y = df["Metro"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [139]:
onehot_encoder = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), ['Births', 'StdBirths']),
    ],
    remainder='passthrough'  # Keep other columns as is
)
model = Pipeline([
    ('preprocessor', onehot_encoder),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [140]:
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)
# Print classification report
print(classification_report(y_test, y_pred))
# Print confusion matrix
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.94      0.88       406
           1       0.86      0.65      0.74       223

    accuracy                           0.84       629
   macro avg       0.85      0.79      0.81       629
weighted avg       0.84      0.84      0.83       629

[[383  23]
 [ 79 144]]
